# Semantic Search & LLM Application
**Prototype asisten customer service e-commerce berbasis FAQ**

Alur besar: **Ingest → Embed → Store → Retrieve → Generate**

| Tahap | Yang terjadi di notebook ini |
|---|---|
| Ingest | Load CSV FAQ, gabungkan question + answer jadi satu unit teks |
| Embed | Ubah tiap unit teks jadi vektor pakai Gemini Embeddings API |
| Store | Simpan semua vektor ke index FAISS (dan ke file `vector_store/faq_index.faiss`) |
| Retrieve | Query user → vektor → cari FAQ paling mirip di FAISS |
| Generate | Masukkan FAQ hasil retrieval ke prompt → Gemini menyusun jawaban |

> Struktur section mengikuti starter notebook dari mentor. Cara menjalankan ada di `README.md`.

## 1. Import Library & Load Knowledge Base

**Kenapa ada `ROOT`?** Notebook ini ada di folder `notebook/`, sedangkan data ada di `data/`.
`ROOT` memastikan path tetap benar, entah notebook dijalankan dari folder `notebook/` atau dari root project.

API key dibaca dari file `.env` (isinya `GEMINI_API_KEY=...`). File ini **tidak** ikut di-commit (ada di `.gitignore`).

In [1]:
import os
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

# Cari root project (folder yang berisi data/)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_PATH = ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv'
INDEX_PATH = ROOT / 'vector_store' / 'faq_index.faiss'

load_dotenv(ROOT / '.env')
if not os.environ.get('GEMINI_API_KEY'):
    raise ValueError('GEMINI_API_KEY not found. Create a .env file in the project root containing GEMINI_API_KEY=<your_key>.')

# timeout 60 detik per request supaya notebook tidak menggantung kalau server Gemini lambat
client = genai.Client(
    api_key=os.environ['GEMINI_API_KEY'],
    http_options=types.HttpOptions(timeout=60_000),
)

# Model yang dipakai. Model di starter notebook (text-embedding-004, gemini-2.0-flash)
# sudah tidak tersedia untuk API key baru, jadi diganti ke versi yang masih aktif.
EMBED_MODEL = 'gemini-embedding-001'
# Urutan prioritas LLM: model pertama dipakai dulu, model kedua jadi cadangan kalau yang pertama
# sedang overload/error. Model "lite" dipilih karena cepat dan cukup untuk menjawab FAQ pendek.
LLM_MODELS = ['gemini-3.5-flash-lite', 'gemini-3.6-flash']

df = pd.read_csv(DATA_PATH)
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


### Cek struktur data
Sebelum diproses, kita lihat dulu: ada kolom apa saja, berapa entri per kategori, dan apakah ada data kosong/duplikat.

In [2]:
print('Shape  :', df.shape)
print('Kolom  :', list(df.columns))
print('Null per kolom:')
print(df.isna().sum().to_string())
print('Duplikat question:', df['question'].duplicated().sum())
print()
print('Jumlah entri per kategori:')
df['category'].value_counts()

Shape  : (59, 4)
Kolom  : ['faq_id', 'category', 'question', 'answer']
Null per kolom:
faq_id      0
category    0
question    0
answer      0
Duplikat question: 0

Jumlah entri per kategori:


category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64

## 2. Menyiapkan Unit Retrievable

Setiap baris FAQ = **satu unit** yang nanti di-embed dan di-retrieve (tidak perlu chunking karena tiap FAQ pendek).

**Kenapa question + answer digabung, bukan question saja?**
- User sering bertanya dengan kata-kata yang lebih mirip ke *jawaban* daripada ke *pertanyaan* FAQ.
  Contoh: "refund masuk ke mana?" punya kata *refund*, yang muncul di answer FAQ #27.
- Kategori juga ikut dimasukkan supaya vektornya membawa konteks topik (Pembayaran, Pengiriman, dst).

Data asli **tidak diubah**. Kita hanya menambah kolom baru `content` di DataFrame (di memori), bukan menulis ulang CSV-nya.

In [3]:
df['content'] = (
    'Kategori: ' + df['category']
    + '\nPertanyaan: ' + df['question']
    + '\nJawaban: ' + df['answer']
)
documents = df['content'].tolist()

print('Jumlah unit retrievable:', len(documents))
print('Contoh unit retrievable:')
print(documents[0])

Jumlah unit retrievable: 59
Contoh unit retrievable:
Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

**Embedding** = representasi teks sebagai deretan angka (vektor). Teks yang maknanya mirip akan menghasilkan vektor yang arahnya mirip,
walaupun kata-katanya beda. Inilah yang bikin *semantic* search beda dari pencarian kata kunci biasa.

Tiga detail penting:
1. **`task_type`**: Gemini bisa mengoptimalkan embedding sesuai perannya.
   Dokumen FAQ di-embed sebagai `RETRIEVAL_DOCUMENT`, query user sebagai `RETRIEVAL_QUERY`.
2. **Batching**: semua dokumen dikirim per batch (bukan 1 request per FAQ), jadi lebih cepat dan hemat kuota.
3. **Normalisasi L2**: tiap vektor dibuat panjangnya = 1. Setelah itu *inner product* antar vektor = **cosine similarity**
   (rentang -1 s/d 1, makin besar makin mirip). Ini dipakai di FAISS pada langkah berikutnya.

`call_with_retry` menangani error rate limit (429) atau server sibuk (5xx) dengan menunggu lalu mencoba lagi.

In [4]:
def call_with_retry(fn, *args, max_retries=3, **kwargs):
    """Panggil fungsi API Gemini; kalau kena rate limit / server error, tunggu lalu coba lagi."""
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except errors.APIError as e:
            if e.code in (429, 500, 503, 504) and attempt < max_retries - 1:
                wait = 5 * 2 ** attempt
                print(f'API error {e.code}, retrying in {wait}s...')
                time.sleep(wait)
            else:
                raise


def embed_texts(texts, task_type, batch_size=50):
    """Ubah list teks jadi matriks vektor float32 yang sudah dinormalisasi (panjang = 1)."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        result = call_with_retry(
            client.models.embed_content,
            model=EMBED_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        vectors.extend(e.values for e in result.embeddings)

    matrix = np.array(vectors, dtype='float32')  # FAISS wajib float32
    faiss.normalize_L2(matrix)                   # in-place: tiap baris jadi panjang 1
    return matrix


doc_embeddings = embed_texts(documents, task_type='RETRIEVAL_DOCUMENT')
print('Shape embeddings:', doc_embeddings.shape)  # (jumlah FAQ, dimensi vektor)

Shape embeddings: (59, 3072)


### Verifikasi embedding sebelum disimpan

In [5]:
n_docs, dimension = doc_embeddings.shape
assert n_docs == len(df), 'Jumlah embedding harus sama dengan jumlah entri FAQ'

norms = np.linalg.norm(doc_embeddings, axis=1)
print('Jumlah vektor   :', n_docs)
print('Dimensi vektor  :', dimension)
print('Dtype           :', doc_embeddings.dtype)
print(f'Panjang vektor  : min={norms.min():.4f}, max={norms.max():.4f} (harus ~1 setelah normalisasi)')
print('Contoh 5 angka pertama vektor FAQ #1:', np.round(doc_embeddings[0][:5], 4))

Jumlah vektor   : 59
Dimensi vektor  : 3072
Dtype           : float32
Panjang vektor  : min=1.0000, max=1.0000 (harus ~1 setelah normalisasi)
Contoh 5 angka pertama vektor FAQ #1: [-0.012   0.0063  0.0214 -0.0712 -0.0025]


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

**FAISS** adalah library untuk menyimpan vektor dan mencari vektor terdekat dengan cepat.

Kita pakai **`IndexFlatIP`** (*Inner Product*):
- *Flat* = membandingkan query ke **semua** vektor (exact search). Untuk 59 FAQ ini sangat cepat dan hasilnya pasti akurat.
- *IP* = skor = inner product. Karena vektor sudah dinormalisasi, skornya = **cosine similarity** (makin besar = makin mirip).

> Starter notebook memakai `IndexFlatL2` (jarak Euclidean, makin kecil = makin mirip). Untuk vektor yang sudah dinormalisasi,
> urutan hasil L2 dan cosine **sama persis**. Cosine dipilih karena skornya lebih mudah dibaca (0 s/d 1).

Urutan vektor di index = urutan baris di `df`. Jadi hasil FAISS berupa posisi `i` → `df.iloc[i]`.

In [6]:
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)
print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(df), 'Jumlah vektor di FAISS tidak sama dengan jumlah FAQ!'

# Simpan index ke file supaya bisa dipakai ulang tanpa embedding ulang
INDEX_PATH.parent.mkdir(exist_ok=True)
faiss.write_index(index, str(INDEX_PATH))

# Cek: baca lagi file index-nya
reloaded = faiss.read_index(str(INDEX_PATH))
print(f'Index disimpan ke {INDEX_PATH.relative_to(ROOT)}, berisi {reloaded.ntotal} vektor')

Jumlah vektor tersimpan di index: 59
Index disimpan ke vector_store\faq_index.faiss, berisi 59 vektor


## 5. Mengimplementasikan Semantic Search

Langkah di dalam `semantic_search`:
1. Query user → embedding (`RETRIEVAL_QUERY`) → dinormalisasi
2. `index.search` → FAISS mengembalikan `top_k` posisi vektor paling mirip + skornya
3. Posisi itu dipetakan balik ke baris FAQ di `df`

In [7]:
def semantic_search(query, top_k=3):
    query_vec = embed_texts([query], task_type='RETRIEVAL_QUERY')  # shape (1, dimension)
    scores, positions = index.search(query_vec, top_k)

    results = []
    for rank, (pos, score) in enumerate(zip(positions[0], scores[0]), start=1):
        row = df.iloc[pos]
        results.append({
            'rank': rank,
            'faq_id': int(row['faq_id']),
            'category': row['category'],
            'question': row['question'],
            'answer': row['answer'],
            'score': round(float(score), 4),  # cosine similarity
        })
    return results


def show_results(results):
    return pd.DataFrame(results)[['rank', 'faq_id', 'category', 'question', 'score']]


# Contoh percobaan
show_results(semantic_search('kenapa paket saya belum sampai?'))

,rank,faq_id,category,question,score
0,1,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.7971
1,2,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.7816
2,3,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.7593


## 6. Mendesain Prompt dan Context

Prompt disusun dengan struktur **Persona, Task, Context, Format**:

| Bagian | Isi |
|---|---|
| **Persona** | Siapa LLM-nya: asisten CS e-commerce yang ramah |
| **Task** | Apa yang harus dikerjakan + aturan: jawab *hanya* dari Context, jangan mengarang |
| **Context** | FAQ hasil semantic search (inilah "pengetahuan" yang boleh dipakai LLM) |
| **Format** | Bentuk jawaban: bahasa, panjang, dan sumber FAQ yang dipakai |

Aturan "kalau tidak ada di Context, bilang tidak tahu" penting karena FAISS **selalu** mengembalikan `top_k` hasil,
bahkan untuk pertanyaan yang tidak ada jawabannya. LLM-lah yang harus menilai apakah FAQ yang diberikan benar-benar menjawab.

In [8]:
FALLBACK_MESSAGE = (
    'Maaf, informasi tersebut belum tersedia di FAQ kami. '
    'Silakan hubungi customer service melalui live chat, email resmi, atau telepon di halaman bantuan.'
)


def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ #{r['faq_id']}] (Kategori: {r['category']})\n"
        f"Pertanyaan: {r['question']}\n"
        f"Jawaban: {r['answer']}"
        for r in search_results
    )

    return f'''### PERSONA
Kamu adalah asisten customer service sebuah platform e-commerce. Gaya bicaramu ramah, sopan, dan to the point.

### TASK
Jawab pertanyaan pelanggan di bawah ini dengan aturan berikut:
1. Gunakan HANYA informasi dari bagian CONTEXT. Jangan memakai pengetahuan umum dan jangan mengarang kebijakan, angka, atau fitur.
2. CONTEXT berisi beberapa FAQ hasil pencarian; tidak semuanya pasti relevan. Pakai hanya FAQ yang benar-benar menjawab pertanyaan.
3. Jika tidak ada FAQ di CONTEXT yang menjawab pertanyaan, balas persis dengan kalimat ini:
   "{FALLBACK_MESSAGE}"

### CONTEXT
{context}

### FORMAT
- Bahasa Indonesia yang natural, 1-4 kalimat. Boleh pakai poin-poin jika ada langkah-langkah.
- Di baris terakhir tulis sumber yang dipakai, contoh: "Sumber: FAQ #12" (atau "Sumber: -" jika memakai pesan fallback).

### PERTANYAAN PELANGGAN
{query}
'''


# Lihat contoh prompt lengkap yang akan dikirim ke LLM
print(build_prompt('apakah bisa bayar pakai cicilan?', semantic_search('apakah bisa bayar pakai cicilan?')))

### PERSONA
Kamu adalah asisten customer service sebuah platform e-commerce. Gaya bicaramu ramah, sopan, dan to the point.

### TASK
Jawab pertanyaan pelanggan di bawah ini dengan aturan berikut:
1. Gunakan HANYA informasi dari bagian CONTEXT. Jangan memakai pengetahuan umum dan jangan mengarang kebijakan, angka, atau fitur.
2. CONTEXT berisi beberapa FAQ hasil pencarian; tidak semuanya pasti relevan. Pakai hanya FAQ yang benar-benar menjawab pertanyaan.
3. Jika tidak ada FAQ di CONTEXT yang menjawab pertanyaan, balas persis dengan kalimat ini:
   "Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan hubungi customer service melalui live chat, email resmi, atau telepon di halaman bantuan."

### CONTEXT
[FAQ #20] (Kategori: Pembayaran)
Pertanyaan: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

[FAQ #17] (Kategori: Pembayaran)
Pertanyaan: Metode pembayaran apa

## 7. Mengintegrasikan LLM

`generate_answer` menggabungkan semuanya jadi satu alur: **query → retrieval → prompt → jawaban**.

Catatan: starter notebook memakai `temperature=0.3`. Untuk model seri Gemini 3, Google menyarankan tetap memakai temperature default,
jadi di sini "jangan mengarang" dijaga lewat **aturan di prompt** (bagian TASK), bukan lewat temperature.

In [9]:
def call_llm(prompt):
    """Kirim prompt ke Gemini. Kalau model utama gagal (overload/timeout), pindah ke model cadangan."""
    last_error = None
    for model in LLM_MODELS:
        try:
            response = call_with_retry(client.models.generate_content, model=model, contents=prompt)
            return response.text.strip()
        except Exception as e:  # APIError atau timeout jaringan
            print(f'Model {model} failed ({type(e).__name__}), trying next model...')
            last_error = e
    raise last_error


def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)   # 1. Retrieve
    prompt = build_prompt(query, search_results)            # 2. Masukkan hasil retrieval ke prompt
    answer = call_llm(prompt)                               # 3. Generate
    return answer, search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang di-retrieve:')
for s in sources:
    print(f"- #{s['faq_id']} (score {s['score']}) {s['question']}")

Jawaban LLM:
Halo Kak! Tentu, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. 

Sumber: FAQ #20

Sumber FAQ yang di-retrieve:
- #20 (score 0.8075) Apakah saya bisa membayar dengan cicilan?
- #17 (score 0.6936) Metode pembayaran apa saja yang tersedia?
- #22 (score 0.6881) Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

7 query uji, dibagi ke 3 tipe sesuai brief. Untuk tiap query ditulis **FAQ yang seharusnya ditemukan** (`expected_faq_id`),
supaya akurasi retrieval bisa diukur, bukan cuma dilihat sekilas:

- **hit@1**: FAQ yang benar ada di peringkat 1
- **hit@3**: FAQ yang benar ada di salah satu dari 3 hasil teratas (ini yang dikirim ke LLM sebagai context)

Query tipe *tidak ada di KB* tidak punya FAQ yang benar (`None`). Yang dinilai di situ adalah apakah LLM memakai pesan fallback.

In [10]:
test_cases = [
    # 1) Jelas ada di knowledge base (kata-katanya mirip FAQ)
    {'type': 'Ada di KB', 'query': 'Metode pembayaran apa saja yang tersedia?', 'expected_faq_id': 17},
    {'type': 'Ada di KB', 'query': 'Bagaimana cara membatalkan pesanan?', 'expected_faq_id': 3},
    # 2) Parafrase / sinonim / bahasa santai: kata kuncinya beda dari FAQ asli
    {'type': 'Parafrase', 'query': 'duit gue kapan balik kalau barangnya udah gue retur?', 'expected_faq_id': 25},
    {'type': 'Parafrase', 'query': 'paket yang dateng udah hancur, harus ngapain?', 'expected_faq_id': 16},
    {'type': 'Parafrase', 'query': 'akun gw ga bisa masuk padahal kemarin masih bisa', 'expected_faq_id': 55},
    # 3) Tidak ada jawabannya di knowledge base
    {'type': 'Tidak ada di KB', 'query': 'Bagaimana resep membuat nasi goreng yang enak?', 'expected_faq_id': None},
    {'type': 'Tidak ada di KB', 'query': 'Apakah ada program afiliasi untuk influencer?', 'expected_faq_id': None},
]

rows = []
for case in test_cases:
    answer, sources = generate_answer(case['query'])
    top_ids = [s['faq_id'] for s in sources]
    expected = case['expected_faq_id']

    rows.append({
        'type': case['type'],
        'query': case['query'],
        'expected_faq_id': expected,
        'top3_faq_ids': top_ids,
        'top1_question': sources[0]['question'],
        'top1_score': sources[0]['score'],
        'hit@1': None if expected is None else top_ids[0] == expected,
        'hit@3': None if expected is None else expected in top_ids,
        'used_fallback': FALLBACK_MESSAGE[:40] in answer,
        'answer': answer,
    })

    print(f"[{case['type']}] {case['query']}")
    for s in sources:
        print(f"   #{s['faq_id']:>2} score={s['score']:.4f} | {s['question']}")
    print('Jawaban:', answer)
    print('-' * 100)
    time.sleep(1)  # jeda kecil supaya tidak kena rate limit

results_df = pd.DataFrame(rows)

[Ada di KB] Metode pembayaran apa saja yang tersedia?
   #17 score=0.7957 | Metode pembayaran apa saja yang tersedia?
   #22 score=0.7261 | Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
   #20 score=0.7062 | Apakah saya bisa membayar dengan cicilan?
Jawaban: Halo! Kami mendukung transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan untuk metode pembayaran yang tersedia. 

Sumber: FAQ #17
----------------------------------------------------------------------------------------------------


[Ada di KB] Bagaimana cara membatalkan pesanan?
   # 3 score=0.7796 | Bagaimana cara membatalkan pesanan yang sudah dibuat?
   #24 score=0.7139 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
   # 8 score=0.7028 | Kenapa pesanan saya otomatis dibatalkan sistem?
Jawaban: Halo! Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama statusnya masih 'Diproses'. Perlu diingat bahwa pembatalan tidak dapat dilakukan setelah status pesanan berubah menjadi 'Dikirim'. 

Sumber: FAQ #3
----------------------------------------------------------------------------------------------------


[Parafrase] duit gue kapan balik kalau barangnya udah gue retur?
   #25 score=0.7815 | Berapa lama proses refund setelah pengembalian disetujui?
   #27 score=0.7315 | Ke mana dana refund akan dikembalikan?
   #24 score=0.7266 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
Jawaban: Halo! Dana refund Anda biasanya akan diproses dalam waktu 3-7 hari kerja setelah barang yang dikembalikan diterima dan diverifikasi oleh penjual. 

Sumber: FAQ #25
----------------------------------------------------------------------------------------------------


[Parafrase] paket yang dateng udah hancur, harus ngapain?
   #16 score=0.8078 | Apa yang harus dilakukan jika paket diterima dalam kondisi rusak?
   #54 score=0.7234 | Saya menerima barang yang salah, apa yang harus saya lakukan?
   #24 score=0.7071 | Bagaimana prosedur pengembalian barang yang tidak sesuai?
Jawaban: Halo! Jika paket diterima dalam kondisi rusak, silakan segera ambil foto atau video kondisi paket saat diterima. Setelah itu, laporkan melalui menu komplain dalam waktu 2x24 jam sejak barang diterima.

Sumber: FAQ #16
----------------------------------------------------------------------------------------------------


[Parafrase] akun gw ga bisa masuk padahal kemarin masih bisa
   #55 score=0.7623 | Kenapa akun saya tiba-tiba tidak bisa login?
   #32 score=0.6973 | Saya lupa password akun, bagaimana cara mengaturnya ulang?
   #35 score=0.6582 | Bagaimana cara menghapus akun secara permanen?
Jawaban: Halo! Akun Anda dapat terkunci sementara karena kesalahan login berulang atau terindikasi aktivitas mencurigakan. Silakan reset password atau hubungi customer service untuk mengatasinya.

Sumber: FAQ #55
----------------------------------------------------------------------------------------------------


[Tidak ada di KB] Bagaimana resep membuat nasi goreng yang enak?
   # 1 score=0.5898 | Bagaimana cara melakukan pemesanan di platform ini?
   #31 score=0.5846 | Bagaimana cara mendaftar akun baru?
   #47 score=0.5767 | Bagaimana cara memberikan ulasan untuk produk yang sudah dibeli?
Jawaban: Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan hubungi customer service melalui live chat, email resmi, atau telepon di halaman bantuan.

Sumber: -
----------------------------------------------------------------------------------------------------


[Tidak ada di KB] Apakah ada program afiliasi untuk influencer?
   #56 score=0.6303 | Apakah member baru mendapat diskon khusus?
   #40 score=0.6278 | Bagaimana cara mendapatkan info promo terbaru?
   #41 score=0.6217 | Apakah cashback dari promo langsung masuk ke saldo?
Jawaban: Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan hubungi customer service melalui live chat, email resmi, atau telepon di halaman bantuan.

Sumber: -
----------------------------------------------------------------------------------------------------


### Ringkasan hasil pengujian

In [11]:
summary = results_df[['type', 'query', 'expected_faq_id', 'top3_faq_ids', 'top1_score', 'hit@1', 'hit@3', 'used_fallback']]
summary

,type,query,expected_faq_id,top3_faq_ids,top1_score,hit@1,hit@3,used_fallback
0,Ada di KB,Metode pembayaran apa saja yang tersedia?,17.0,"[17, 22, 20]",0.7957,True,True,False
1,Ada di KB,Bagaimana cara membatalkan pesanan?,3.0,"[3, 24, 8]",0.7796,True,True,False
2,Parafrase,duit gue kapan balik kalau barangnya udah gue ...,25.0,"[25, 27, 24]",0.7815,True,True,False
3,Parafrase,"paket yang dateng udah hancur, harus ngapain?",16.0,"[16, 54, 24]",0.8078,True,True,False
4,Parafrase,akun gw ga bisa masuk padahal kemarin masih bisa,55.0,"[55, 32, 35]",0.7623,True,True,False
5,Tidak ada di KB,Bagaimana resep membuat nasi goreng yang enak?,NaN,"[1, 31, 47]",0.5898,None,None,True
6,Tidak ada di KB,Apakah ada program afiliasi untuk influencer?,NaN,"[56, 40, 41]",0.6303,None,None,True


In [12]:
answerable = results_df[results_df['expected_faq_id'].notna()]
unanswerable = results_df[results_df['expected_faq_id'].isna()]

print(f"hit@1 (query yang ada jawabannya): {answerable['hit@1'].sum()}/{len(answerable)}")
print(f"hit@3 (query yang ada jawabannya): {answerable['hit@3'].sum()}/{len(answerable)}")
print(f"Fallback dipakai pada query tanpa jawaban: {unanswerable['used_fallback'].sum()}/{len(unanswerable)}")
print(f"Fallback (salah) dipakai pada query yang ada jawabannya: {answerable['used_fallback'].sum()}/{len(answerable)}")
print()
print('Rata-rata top-1 score per tipe query:')
print(results_df.groupby('type', sort=False)['top1_score'].mean().round(4).to_string())

hit@1 (query yang ada jawabannya): 5/5
hit@3 (query yang ada jawabannya): 5/5
Fallback dipakai pada query tanpa jawaban: 2/2
Fallback (salah) dipakai pada query yang ada jawabannya: 0/5

Rata-rata top-1 score per tipe query:
type
Ada di KB          0.7876
Parafrase          0.7839
Tidak ada di KB    0.6100


### Simpan hasil pengujian ke `documentation/`
Hasil mentah (query, FAQ yang di-retrieve, skor, jawaban LLM) disimpan ke `documentation/retrieval_test_results.md`.
Analisisnya ada di bagian berikut dan di `README.md`.

In [13]:
lines = ['# Retrieval Test Results', '',
         f'Embedding model: `{EMBED_MODEL}` | LLM: `{LLM_MODELS[0]}` (cadangan: `{LLM_MODELS[1]}`) | top_k = 3 | skor = cosine similarity', '',
         '| # | Tipe | Query | Expected | Top-3 FAQ | Top-1 score | hit@1 | hit@3 | Fallback |',
         '|---|---|---|---|---|---|---|---|---|']
for i, r in results_df.iterrows():
    exp = '-' if r['expected_faq_id'] is None or pd.isna(r['expected_faq_id']) else f"#{int(r['expected_faq_id'])}"
    fmt = lambda v: '-' if v is None else ('✅' if v else '❌')
    lines.append(f"| {i + 1} | {r['type']} | {r['query']} | {exp} | {', '.join(f'#{x}' for x in r['top3_faq_ids'])} "
                 f"| {r['top1_score']:.4f} | {fmt(r['hit@1'])} | {fmt(r['hit@3'])} | {'ya' if r['used_fallback'] else 'tidak'} |")

lines += ['', '## Jawaban LLM per query', '']
for i, r in results_df.iterrows():
    lines += [f"### {i + 1}. {r['query']}", f"*Tipe: {r['type']}*", '', r['answer'], '']

out_path = ROOT / 'documentation' / 'retrieval_test_results.md'
out_path.parent.mkdir(exist_ok=True)
out_path.write_text('\n'.join(lines), encoding='utf-8')
print('Disimpan ke', out_path.relative_to(ROOT))

Disimpan ke documentation\retrieval_test_results.md


### Catatan Analisis Relevansi

> Hasil di bawah berdasarkan run notebook ini (`gemini-embedding-001`, LLM `gemini-3.5-flash-lite`, top_k = 3, skor = cosine similarity).
> Jawaban LLM bisa sedikit berbeda tiap kali dijalankan, tetapi hasil retrieval (FAISS) akan sama selama embedding-nya sama.

**1. Retrieval akurat untuk semua query yang ada jawabannya (hit@1 = 5/5).**
FAQ yang benar selalu ada di peringkat 1, dengan skor top-1 sekitar 0.76–0.81.

**2. Query parafrase/bahasa santai dapat skor yang hampir sama dengan query yang kata-katanya mirip FAQ** (rata-rata 0.784 vs 0.788).
Ini bukti semantic search menangkap *makna*, bukan sekadar kata kunci:
- "duit gue kapan **balik** … udah gue **retur**" → FAQ #25 *"Berapa lama proses **refund** setelah **pengembalian** disetujui?"*. Tidak ada kata yang sama persis.
- "paket … udah **hancur**, harus **ngapain**?" → FAQ #16 *"Apa yang harus dilakukan jika paket diterima dalam kondisi **rusak**?"*
- "akun gw **ga bisa masuk**" → FAQ #55 *"Kenapa akun saya tiba-tiba tidak bisa **login**?"*

Kalau pakai pencarian kata kunci biasa, query-query ini kemungkinan besar gagal.

**3. Hasil peringkat 2–3 sering "topiknya dekat, tapi bukan jawabannya".**
Contoh: "cara membatalkan pesanan" juga menarik FAQ #24 (pengembalian barang, 0.71) dan #8 (pesanan dibatalkan otomatis oleh sistem, 0.70).
Selisih skornya dengan FAQ yang benar cuma ~0.07. Di sini LLM berperan penting: prompt menyuruh LLM memakai FAQ yang benar-benar relevan saja,
dan di semua kasus LLM hanya mengutip FAQ yang tepat (lihat baris "Sumber: FAQ #…").

**4. Query yang tidak ada di knowledge base tetap mengembalikan 3 hasil, tetapi skornya lebih rendah.**
- "resep nasi goreng" → skor 0.59. FAQ yang terambil (#1, #31, #47) hanya mirip *pola kalimatnya* ("Bagaimana cara …"), bukan topiknya.
- "program afiliasi influencer" → skor 0.63. Yang terambil FAQ seputar promo/diskon, topiknya berdekatan tapi tidak menjawab.

FAISS **tidak bisa bilang "tidak tahu"**: dia selalu mengembalikan top_k terdekat. Yang menjaga sistem agar tidak mengarang adalah **aturan di prompt**.
Di kedua kasus, LLM memakai pesan fallback dengan benar (2/2), dan tidak ada query yang seharusnya bisa dijawab malah dijawab "tidak tahu" (0/5).

**5. Kasus yang paling berisiko adalah query yang topiknya berdekatan dengan FAQ** (seperti "afiliasi" vs promo).
Selisih skornya dengan query yang bisa dijawab (0.63 vs ±0.76) jauh lebih tipis dibanding query yang benar-benar di luar topik (0.59).
Kalau prompt-nya longgar, di sini LLM paling mungkin "mengarang" jawaban dari FAQ promo.

**Kesimpulan:** retrieval bekerja baik untuk query langsung maupun parafrase. Penanganan pertanyaan di luar knowledge base saat ini sepenuhnya bergantung pada prompt.
Perbaikan berikutnya adalah menambahkan **ambang skor** (misalnya < 0.68 → langsung fallback tanpa memanggil LLM). Nilai ambang ini perlu divalidasi dengan
lebih banyak query uji, karena 7 query belum cukup untuk menentukannya dengan yakin.